# factor-pit-cash-accrual-level-v01

Local candidate package.

## AI application and current local status

- **local RED/GREEN and main-only checks passed:** production and package
  contracts, including local main-only isolation, were executed locally.
- **T1/T2 training compatibility passed:** both archived checks prove
  training-environment compatibility only.
- The **official financial lookahead path is not publicly exposed**, and **no official financial lookahead pass is claimed**.
- **no candidate performance result has been read:** no returns, IC,
  ModelScore, leaderboard, effect/evaluator, or simulation result was read.
- This package is **ready for separate formal-submission authorization** and **has not been submitted**.

This deterministic local artifact has
`competition_submission_status=submission_package_ready_pending_formal_submission_authorization`.


In [ ]:
# AI application evidence is recorded in ai-application-notes.md.
# This passive cell has no imports or calls.

In [ ]:
def main(datasources, start_date, end_date):
    """Return the PIT cash-accrual-level factor for the requested natural dates."""
    import numpy as np
    import pandas as pd
    import dai

    requested_start = pd.Timestamp(start_date)
    requested_end = pd.Timestamp(end_date)
    if pd.isna(requested_start) or pd.isna(requested_end) or requested_end < requested_start:
        raise ValueError("requested dates are invalid")
    requested_start_day = requested_start.normalize()
    requested_end_day = requested_end.normalize()
    query_start_date = requested_start - pd.Timedelta(days=365)
    query_start_day = query_start_date.normalize()
    query_start_text = query_start_date.strftime("%Y-%m-%d %H:%M:%S")
    requested_end_text = requested_end.strftime("%Y-%m-%d %H:%M:%S")
    financial_table = datasources["financial"]

    financial_sql = f"""
    -- PIT_CASH_ACCRUAL_LEVEL_FACTOR_QUERY
    WITH cte_ttm AS (
        SELECT
            date,
            instrument,
            net_profit_to_parent_shareholders AS np_ttm,
            net_cffoa AS cffoa_ttm
        FROM {financial_table}
        WHERE category = 'ttm'
          AND shift = 0
          AND date >= '{query_start_text}'
          AND date <= '{requested_end_text}'
    ),
    cte_lf AS (
        SELECT
            date,
            instrument,
            total_assets AS assets_lf
        FROM {financial_table}
        WHERE category = 'lf'
          AND shift = 0
          AND date >= '{query_start_text}'
          AND date <= '{requested_end_text}'
    )
    SELECT
        ttm.date,
        ttm.instrument,
        ttm.np_ttm AS np_ttm,
        ttm.cffoa_ttm AS cffoa_ttm,
        lf.assets_lf AS assets_lf
    FROM cte_ttm ttm
    PRUNE JOIN cte_lf lf
      ON ttm.date = lf.date
     AND ttm.instrument = lf.instrument
    """
    financial = dai.query(
        financial_sql,
        filters={"date": [query_start_text, requested_end_text]},
        compression=True,
    ).df().copy()
    required_financial_columns = {"date", "instrument", "np_ttm", "cffoa_ttm", "assets_lf"}
    missing_financial_columns = required_financial_columns.difference(financial.columns)
    if missing_financial_columns:
        raise ValueError(f"financial query missing columns: {sorted(missing_financial_columns)}")
    financial["date"] = pd.to_datetime(financial["date"], errors="coerce").dt.normalize()
    if financial["date"].isna().any():
        raise ValueError("financial query has invalid dates")
    financial_instruments = financial["instrument"].astype("string")
    if financial_instruments.isna().any() or financial_instruments.str.strip().eq("").any():
        raise ValueError("financial query has invalid instruments")
    financial["instrument"] = financial_instruments.str.strip().astype(object)
    if financial.duplicated(["date", "instrument"]).any():
        raise ValueError("duplicate financial announcement keys")
    financial = financial.loc[
        financial["date"].between(query_start_day, requested_end_day)
    ].copy()
    for column in ("np_ttm", "cffoa_ttm", "assets_lf"):
        financial[column] = pd.to_numeric(financial[column], errors="coerce")

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df().copy()
    required_pool_columns = {"date", "instrument"}
    missing_pool_columns = required_pool_columns.difference(pool.columns)
    if missing_pool_columns:
        raise ValueError(f"stock pool missing columns: {sorted(missing_pool_columns)}")
    pool["date"] = pd.to_datetime(pool["date"], errors="coerce").dt.normalize()
    if pool["date"].isna().any():
        raise ValueError("stock pool has invalid dates")
    pool_instruments = pool["instrument"].astype("string")
    if pool_instruments.isna().any() or pool_instruments.str.strip().eq("").any():
        raise ValueError("stock pool has invalid instruments")
    pool["instrument"] = pool_instruments.str.strip().astype(object)
    if pool.duplicated(["date", "instrument"]).any():
        raise ValueError("duplicate stock pool keys")
    pool = pool.loc[pool["date"].between(requested_start_day, requested_end_day)].copy()

    calendar = pd.date_range(query_start_day, requested_end_day, freq="D")
    financial = financial.sort_values(["instrument", "date"]).reset_index(drop=True)
    financial["_event_id"] = np.arange(len(financial), dtype=np.int64)
    expanded_frames = []
    for instrument, events in financial.groupby("instrument", sort=False):
        daily = events.set_index("date")[["_event_id"]].reindex(calendar)
        daily["_event_id"] = daily["_event_id"].ffill()
        daily = daily.reset_index(names="date")
        daily["instrument"] = instrument
        expanded_frames.append(daily)
    if expanded_frames:
        daily = pd.concat(expanded_frames, ignore_index=True)
        daily = daily.merge(
            financial[["_event_id", "np_ttm", "cffoa_ttm", "assets_lf"]],
            on="_event_id",
            how="left",
            validate="many_to_one",
        )
    else:
        daily = pd.DataFrame(
            {
                "date": pd.Series(dtype="datetime64[ns]"),
                "instrument": pd.Series(dtype=object),
                "np_ttm": pd.Series(dtype=float),
                "cffoa_ttm": pd.Series(dtype=float),
                "assets_lf": pd.Series(dtype=float),
            }
        )
    eligible = (
        np.isfinite(daily["np_ttm"])
        & np.isfinite(daily["cffoa_ttm"])
        & np.isfinite(daily["assets_lf"])
        & (daily["assets_lf"] > 0)
    )
    daily = daily.loc[eligible, ["date", "instrument", "np_ttm", "cffoa_ttm", "assets_lf"]].copy()
    daily["factor"] = -(daily["np_ttm"] - daily["cffoa_ttm"]) / daily["assets_lf"]
    daily = daily.loc[np.isfinite(daily["factor"]), ["date", "instrument", "factor"]]

    merged = daily.merge(pool, on=["date", "instrument"], how="right", validate="one_to_one")
    result = merged.loc[np.isfinite(merged["factor"]), ["date", "instrument", "factor"]].copy()
    result = result.loc[result["date"].between(requested_start_day, requested_end_day)].copy()
    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("duplicate output keys")
    if not np.isfinite(result["factor"]).all():
        raise ValueError("output factor must be finite")
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
